# Formal Layer2R one-case regression

This runtime-only wrapper verifies CUDA, resolves the attached dataset, runs the formal preflight, and executes at most one pending locked case.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import torch

REPOSITORY_MARKERS = ("src", "experiments", "configs")


def is_repository_root(path: Path) -> bool:
    return all((path / marker).is_dir() for marker in REPOSITORY_MARKERS)


current_directory = Path.cwd().resolve()
if is_repository_root(current_directory):
    REPOSITORY_ROOT = current_directory
else:
    repository_roots = {
        parent.resolve() for parent in current_directory.parents if is_repository_root(parent)
    }
    kaggle_working = Path("/kaggle/working")
    if kaggle_working.is_dir():
        if is_repository_root(kaggle_working):
            repository_roots.add(kaggle_working.resolve())
        repository_roots.update(
            path.resolve()
            for path in kaggle_working.rglob("*")
            if path.is_dir() and is_repository_root(path)
        )
    if not repository_roots:
        raise RuntimeError("Could not find a repository root containing src/, experiments/, and configs/")
    if len(repository_roots) != 1:
        candidates = ", ".join(str(path) for path in sorted(repository_roots))
        raise RuntimeError(f"Found multiple repository roots containing src/, experiments/, and configs/: {candidates}")
    REPOSITORY_ROOT = repository_roots.pop()

sys.path.insert(0, str(REPOSITORY_ROOT))

from src.pipelines.formal_layer2r import load_formal_config

CONFIG_PATH = REPOSITORY_ROOT / "configs/layer2r_kaggle_one_case.json"
RUNNER_PATH = REPOSITORY_ROOT / "experiments/run_formal_layer2r.py"

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for the formal one-case regression")
print("CUDA device:", torch.cuda.get_device_name(0))

with CONFIG_PATH.open() as stream:
    raw_config = json.load(stream)
if raw_config.get("max_new_cases") != 1:
    raise RuntimeError("One-case wrapper requires max_new_cases == 1")

config = load_formal_config(CONFIG_PATH)
print("Resolved dataset root:", config.raw_root)

command = [sys.executable, str(RUNNER_PATH), "--config", str(CONFIG_PATH)]
preflight = subprocess.run([*command, "--preflight"], check=False)
if preflight.returncode != 0:
    raise RuntimeError(f"Formal one-case preflight failed with exit code {preflight.returncode}")

execution = subprocess.run(command, check=False)
if execution.returncode != 0:
    raise RuntimeError(f"Formal one-case execution failed with exit code {execution.returncode}")
print("Formal Layer2R one-case regression completed; wrapper stopping.")
